In [3]:
import pandas as pd
import numpy as np
import os
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# Load data
csv_path = 'all_temperature_cleaned.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
df['datetime'] = pd.to_datetime(df['timestamp'] + ' ' + df['time'], format='%Y-%m-%d %H:%M')

regions = ['Rakhiyal', 'Bopal', 'Ambawadi', 'Chandkheda', 'Vastral']

# Split train (2019-2023) and test (2024)
train_df = df[(df['datetime'].dt.year >= 2019) & (df['datetime'].dt.year <= 2023)].copy()
test_df = df[df['datetime'].dt.year == 2024].copy()

output_folder_2025 = 'predictions_2025'
os.makedirs(output_folder_2025, exist_ok=True)

current_dir = os.getcwd()
metrics_list = []

seasonal_periods_options = [24, 168]  # daily and weekly seasonality
seasonal_types = ['add', 'mul']
trend_types = ['add', 'mul', None]

for region in regions:
    print(f"\nOptimizing Holt-Winters for region: {region}")

    train_series = train_df.set_index('datetime')[region].interpolate(method='time')
    test_series = test_df.set_index('datetime')[region].interpolate(method='time')

    best_rmse = float('inf')
    best_model = None
    best_params = None

    for sp in seasonal_periods_options:
        for seasonal in seasonal_types:
            for trend in trend_types:
                try:
                    model = ExponentialSmoothing(
                        train_series,
                        trend=trend,
                        seasonal=seasonal,
                        seasonal_periods=sp,
                        initialization_method='estimated'
                    )
                    fit = model.fit(optimized=True)
                    preds = fit.forecast(len(test_series))
                    rmse = np.sqrt(mean_squared_error(test_series, preds))
                    print(f"SP={sp}, Seasonal={seasonal}, Trend={trend}, RMSE={rmse:.4f}")
                    if rmse < best_rmse:
                        best_rmse = rmse
                        best_model = fit
                        best_params = {'seasonal_periods': sp, 'seasonal': seasonal, 'trend': trend}
                except Exception:
                    continue

    print(f"Best params for {region}: {best_params} with RMSE={best_rmse:.4f}")

    # Forecast 2024 with best model
    pred_2024 = best_model.forecast(len(test_series))

    pred_2024_df = pd.DataFrame({
        'date': test_series.index.date,
        'hour': test_series.index.hour,
        'predicted_temperature': pred_2024.values,
        'actual_temperature': test_series.values
    })
    filename_2024 = f"holtwinters_{region.lower()}_2024.csv"
    pred_2024_df.to_csv(os.path.join(current_dir, filename_2024), index=False)
    print(f"Saved 2024 predictions for {region} as {filename_2024}")

    # Forecast 2025 hourly
    date_range_2025 = pd.date_range(start='2025-01-01 00:00', end='2025-12-31 23:00', freq='H')
    pred_2025 = best_model.forecast(len(date_range_2025))

    pred_2025_df = pd.DataFrame({
        'date': date_range_2025.date,
        'hour': date_range_2025.hour,
        'predicted_temperature': pred_2025.values
    })
    filename_2025 = f"holtwinters_{region.lower()}_2025.csv"
    pred_2025_df.to_csv(os.path.join(output_folder_2025, filename_2025), index=False)
    print(f"Saved 2025 predictions for {region} as {filename_2025}")

    metrics_list.append({
        'region': region,
        'rmse_2024': best_rmse,
        'best_params': best_params
    })

metrics_df = pd.DataFrame(metrics_list)
metrics_filename = 'holtwinters_model_metrics_2024.csv'
metrics_df.to_csv(os.path.join(current_dir, metrics_filename), index=False)
print(f"\nSaved all regions tuning metrics as {metrics_filename}")



Optimizing Holt-Winters for region: Rakhiyal
SP=24, Seasonal=add, Trend=add, RMSE=13.1973
SP=24, Seasonal=add, Trend=mul, RMSE=47.4485
SP=24, Seasonal=add, Trend=None, RMSE=9.6444
SP=24, Seasonal=mul, Trend=add, RMSE=15.7121
SP=24, Seasonal=mul, Trend=mul, RMSE=9.8450
SP=24, Seasonal=mul, Trend=None, RMSE=8.7548
SP=168, Seasonal=add, Trend=add, RMSE=28.2338
SP=168, Seasonal=add, Trend=mul, RMSE=17.9870
SP=168, Seasonal=add, Trend=None, RMSE=9.2094
SP=168, Seasonal=mul, Trend=add, RMSE=26.8002
SP=168, Seasonal=mul, Trend=mul, RMSE=8.4173
SP=168, Seasonal=mul, Trend=None, RMSE=10.4417
Best params for Rakhiyal: {'seasonal_periods': 168, 'seasonal': 'mul', 'trend': 'mul'} with RMSE=8.4173
Saved 2024 predictions for Rakhiyal as holtwinters_hourly_rakhiyal_2024.csv
Saved 2025 predictions for Rakhiyal as holtwinters_hourly_rakhiyal_2025.csv

Optimizing Holt-Winters for region: Bopal
SP=24, Seasonal=add, Trend=add, RMSE=20.3911
SP=24, Seasonal=add, Trend=mul, RMSE=48.3481
SP=24, Seasonal=add,